# Audit mixed Born response

Audits the locally available mixed-curvature and Born-charge validations. Deliberately separates executable pass/fail checks from accuracy gates that require new first-principles data. Reads the canonical NPZ artifacts produced by `born_effective_charge.ipynb` and writes one compact JSON summary; does not recompute the expensive Brillouin-zone integrals itself.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved `results/mixed_born/mixed_born_validation_summary.json`.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS, STO, STO_BASE_RUN
from validation._paths import DATA
from validation.observable.born_effective_charge_lib import (
    centre_cross_check,
    parse_ph_born_charges,
)

## Config

In [ ]:
RUN_COMPUTATION = False  # True: reload every source born_charge npz and recompute the summary. False: load results/mixed_born/mixed_born_validation_summary.json.

TI_BORN = (DATA / "SrTiO3/Ti_Q_01A/output/181839bc889/trial_03_Sr_sp_Ti_pd_Osp/no_displacement/185119bc889" / "born_charge")
TI_OCC_BORN = (DATA / "SrTiO3/Ti_Q_01A/output/181839bc889/trial_02_Sr_p_O_sp/no_displacement/183965bc889" / "born_charge")
SR_BORN = (DATA / "SrTiO3/Sr_Q_01A/output/184367bc889/trial_03_Sr_sp_Ti_pd_O_sp/no_displacement/188444bc889" / "born_charge")
SR_OCC_BORN = (DATA / "SrTiO3/Sr_Q_01A/output/184367bc889/trial_02_Sr_p_O_sp/no_displacement/188438bc889" / "born_charge")
BASE_OCC = DATA / "SrTiO3/base/soc/output/181642bc889/trial_02_Sr_p_O_sp/183813bc889"
TI_OCC_MODE = TI_OCC_BORN.parent
SR_OCC_MODE = SR_OCC_BORN.parent
PH_OUT = STO_BASE_RUN / "SrTiO3.ph.out"
OUTPUT = RESULTS / "mixed_born/mixed_born_validation_summary.json"

## Helpers

In [ ]:
def load(directory: Path, filename: str) -> dict:
    path = directory / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as archive:
        return {key: archive[key] for key in archive.files}


def index_for_nk(data: dict, nk: int) -> int:
    matches = np.flatnonzero(data["nks"] == nk)
    if len(matches) != 1:
        raise ValueError(f"expected one nk={nk} row, found {len(matches)}")
    return int(matches[0])


def scalar_at(data: dict, key: str, nk: int, axis: int = 2) -> float:
    return float(data[key][index_for_nk(data, nk), axis])


def benchmark(data: dict, nk: int = 16) -> dict:
    value = scalar_at(data, "z_total", nk)
    reference = float(data["z_qe"][2])
    error = value - reference
    transverse = data["z_total"][index_for_nk(data, nk), :2]
    return {
        "nk": nk, "wannier": value, "DFPT": reference, "signed_error": error,
        "relative_error": abs(error / reference),
        "max_abs_symmetry_forbidden_zx_zy": float(np.max(np.abs(transverse))),
        "passes_one_percent_accuracy": bool(abs(error / reference) < 0.01),
        "passes_transverse_symmetry_1e-7": bool(np.max(np.abs(transverse)) < 1e-7),
    }


def convergence(data: dict, left: int = 12, right: int = 16) -> dict:
    z_left = scalar_at(data, "z_total", left)
    z_right = scalar_at(data, "z_total", right)
    difference = abs(z_right - z_left)
    return {
        "nk_pair": [left, right], "values": [z_left, z_right],
        "absolute_change": difference, "relative_change": difference / abs(z_right),
        "passes_5e-4_absolute": bool(difference < 5e-4),
    }

## Load every source NPZ

13 saved `born_effective_charge` runs: Ti/Sr, full-space/occupied-only, several flag combinations.

In [ ]:
if RUN_COMPUTATION:
    ti = load(TI_BORN, "born_charge_Ti_Q_01A_mmn_taucentres_mhn0_wbnone.npz")
    ti_tau_zero = load(TI_BORN, "born_charge_Ti_Q_01A_mmn_tauzero_mhn0_wbnone.npz")
    ti_zero_displacement = load(TI_BORN, "born_charge_self_test_mmn_taucentres_mhn0_wbnone.npz")
    ti_default = load(TI_BORN, "born_charge_Ti_Q_01A_mmn_taucentres_wbnone.npz")
    ti_rfull = load(TI_BORN, "born_charge_Ti_Q_01A_rfull_taucentres.npz")
    ti_occ = load(TI_OCC_BORN, "born_charge_Ti_Q_01A_mmn_taucentres_wbnone.npz")
    ti_occ_rfull = load(TI_OCC_BORN, "born_charge_Ti_Q_01A_rfull_taucentres.npz")
    ti_semicore = load(TI_OCC_BORN, "born_charge_semicore_mmn_taucentres_wbnone.npz")
    sr = load(SR_BORN, "born_charge_Sr_Q_01A_mmn_taucentres_mhn0_wbnone.npz")
    sr_default = load(SR_BORN, "born_charge_Sr_Q_01A_mmn_taucentres_wbnone.npz")
    sr_rfull = load(SR_BORN, "born_charge_Sr_Q_01A_rfull_taucentres.npz")
    sr_occ = load(SR_OCC_BORN, "born_charge_Sr_Q_01A_mmn_taucentres_wbnone.npz")
    sr_occ_rfull = load(SR_OCC_BORN, "born_charge_Sr_Q_01A_rfull_taucentres.npz")

## Embedding covariance: tau=centres vs tau=zero

In [ ]:
if RUN_COMPUTATION:
    common_nks = sorted(set(ti["nks"].tolist()) & set(ti_tau_zero["nks"].tolist()))
    centre_total = np.stack([ti["z_total"][index_for_nk(ti, nk)] for nk in common_nks])
    zero_total = np.stack([ti_tau_zero["z_total"][index_for_nk(ti_tau_zero, nk)] for nk in common_nks])
    embedding_total_difference = float(np.max(np.abs(centre_total - zero_total)))
    embedding_part_redistribution = max(
        float(np.max(np.abs(ti[key] - ti_tau_zero[key]))) for key in ("z_int", "z_cross", "z_ext")
    )
    print(f"embedding_total_difference={embedding_total_difference:.2e}, "
          f"part_redistribution={embedding_part_redistribution:.2e}")

## Occupied-only exact reductions: curvature integral vs Wannier-centre formula

In [ ]:
if RUN_COMPUTATION:
    ti_centres = centre_cross_check(BASE_OCC, TI_OCC_MODE, 0.01)
    sr_centres = centre_cross_check(BASE_OCC, SR_OCC_MODE, 0.01)
    ti_occ_curvature = ti_occ["z_el"][index_for_nk(ti_occ, 16)]
    sr_occ_curvature = sr_occ["z_el"][index_for_nk(sr_occ, 16)]
    print("Ti curvature vs centres, max abs diff:", float(np.max(np.abs(ti_occ_curvature - ti_centres))))
    print("Sr curvature vs centres, max abs diff:", float(np.max(np.abs(sr_occ_curvature - sr_centres))))

## DFPT acoustic sum rule

In [ ]:
if RUN_COMPUTATION:
    raw_dfpt = parse_ph_born_charges(PH_OUT, asr=False)
    asr_dfpt = parse_ph_born_charges(PH_OUT, asr=True)
    raw_sum = sum((tensor for _, tensor in raw_dfpt.values()), np.zeros((3, 3)))
    asr_sum = sum((tensor for _, tensor in asr_dfpt.values()), np.zeros((3, 3)))
    print("raw ASR violation, max abs:", float(np.max(np.abs(raw_sum))))
    print("ASR-applied residual, max abs:", float(np.max(np.abs(asr_sum))))

## Position-discretization sensitivity: mmn vs rfull

In [ ]:
if RUN_COMPUTATION:
    ti_scheme_difference = scalar_at(ti_rfull, "z_total", 16) - scalar_at(ti_default, "z_total", 16)
    sr_scheme_difference = scalar_at(sr_rfull, "z_total", 16) - scalar_at(sr_default, "z_total", 16)
    ti_occ_scheme_difference = scalar_at(ti_occ_rfull, "z_total", 16) - scalar_at(ti_occ, "z_total", 16)
    sr_occ_scheme_difference = scalar_at(sr_occ_rfull, "z_total", 16) - scalar_at(sr_occ, "z_total", 16)

    second_ti_amplitude_present = any(
        candidate.exists() for candidate in (STO / "Ti_Q_005A", STO / "Ti_Q_0p005A", STO / "Ti_Q_05A")
    )
    oxygen_displacement_present = any(STO.glob("O*_Q_*A"))

## Assemble the summary

In [ ]:
if RUN_COMPUTATION:
    summary = {
        "zero_displacement": {
            "max_abs_electronic_response": float(np.max(np.abs(ti_zero_displacement["z_el"]))),
            "passes_1e-12": bool(np.max(np.abs(ti_zero_displacement["z_el"])) < 1e-12),
        },
        "embedding_covariance": {
            "nks": common_nks,
            "max_abs_total_difference": embedding_total_difference,
            "max_abs_redistribution_among_parts": embedding_part_redistribution,
            "passes_total_1e-10": bool(embedding_total_difference < 1e-10),
        },
        "DFPT_benchmarks": {"Ti_trial03": benchmark(ti), "Sr_trial03": benchmark(sr)},
        "interpolation_kmesh_convergence": {"Ti_trial03": convergence(ti), "Sr_trial03": convergence(sr)},
        "occupied_only_exact_reductions": {
            "Ti_curvature_vs_centre_max_abs": float(np.max(np.abs(ti_occ_curvature - ti_centres))),
            "Sr_curvature_vs_centre_max_abs": float(np.max(np.abs(sr_occ_curvature - sr_centres))),
            "Ti_mmn_vs_rfull_Zstar_difference": ti_occ_scheme_difference,
            "Sr_mmn_vs_rfull_Zstar_difference": sr_occ_scheme_difference,
            "passes_centre_formula_5e-3": bool(
                max(np.max(np.abs(ti_occ_curvature - ti_centres)), np.max(np.abs(sr_occ_curvature - sr_centres))) < 5e-3
            ),
            "passes_position_scheme_cancellation_1e-6": bool(
                max(abs(ti_occ_scheme_difference), abs(sr_occ_scheme_difference)) < 1e-6
            ),
        },
        "semicore_null": {
            "nk": int(ti_semicore["nks"][-1]), "z_el": ti_semicore["z_el"][-1].tolist(),
            "max_abs": float(np.max(np.abs(ti_semicore["z_el"][-1]))),
            "passes_calibrated_0p5_tolerance": bool(np.max(np.abs(ti_semicore["z_el"][-1])) < 0.5),
        },
        "position_discretization_sensitivity": {
            "Ti_trial03_rfull_minus_mmn_Zstar": ti_scheme_difference,
            "Sr_trial03_rfull_minus_mmn_Zstar": sr_scheme_difference,
            "interpretation": "diagnostic, not a pass/fail check; the two finite-8^3 connection discretizations need not coincide",
        },
        "DFPT_acoustic_sum_rule": {
            "raw_max_abs_sum": float(np.max(np.abs(raw_sum))), "asr_applied_max_abs_sum": float(np.max(np.abs(asr_sum))),
            "raw_passes_5e-3": bool(np.max(np.abs(raw_sum)) < 5e-3), "asr_applied_passes_1e-4": bool(np.max(np.abs(asr_sum)) < 1e-4),
        },
        "unavailable_accuracy_gates": {
            "second_Ti_displacement_0p005A": {
                "status": "available" if second_ti_amplitude_present else "NOT_RUN",
                "reason": "needed to remove/check the O(delta_u) forward-difference error",
            },
            "interpolated_acoustic_sum_rule_with_O_displacement": {
                "status": "available" if oxygen_displacement_present else "NOT_RUN",
                "reason": "no O-displaced Wannier endpoint is present locally",
            },
            "non_frozen_Lambda_R": {
                "status": "NOT_RUN",
                "reason": "all stored responses use the explicitly frozen projection gauge; no independently computed Lambda(R) is present",
            },
        },
    }
    pass_flags = []
    for section in (
        summary["zero_displacement"], summary["embedding_covariance"],
        *summary["DFPT_benchmarks"].values(), *summary["interpolation_kmesh_convergence"].values(),
        summary["occupied_only_exact_reductions"], summary["semicore_null"], summary["DFPT_acoustic_sum_rule"],
    ):
        pass_flags.extend(value for key, value in section.items() if key.startswith("passes_"))
    summary["all_locally_executable_checks_pass"] = bool(all(pass_flags))
    summary["production_accuracy_gate_complete"] = bool(
        all(item["status"] == "available" for item in summary["unavailable_accuracy_gates"].values())
    )

## Save

In [ ]:
if RUN_COMPUTATION:
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print(f"saved {OUTPUT}")

## Load a previous run (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    summary = json.loads(OUTPUT.read_text())
    print(f"loaded {OUTPUT}")

## Inspect

In [ ]:
print(json.dumps(summary, indent=2, sort_keys=True))
print("\nall_locally_executable_checks_pass:", summary["all_locally_executable_checks_pass"])
print("production_accuracy_gate_complete:", summary["production_accuracy_gate_complete"])